# Final dataset analysis

This notebook summarizes facility/state coverage, NAICS diversity, and field completeness for the final extracted dataset.

## Load data

In [1]:
import os

import numpy as np
import pandas as pd

from permit_data_extraction.config import PROCESSED_DATA_DIR

2026-02-06 16:04:33.118 | INFO     | permit_data_extraction.config:<module>:11 - PROJ_ROOT path is: /home/afhubbard/permit_data_extraction


In [2]:
FILE_PATH = os.path.join(PROCESSED_DATA_DIR, "permit_data_extracted.xlsx")

# Read the final dataset
raw_df = pd.read_excel(FILE_PATH)
raw_df.head()

,Filename,Status,Processing Date,Facility Name,Facility Address,Facility City,Facility State Abbreviation,Facility Zip Code,Facility County,NAICS Code,...,Capacity Value,Capacity Unit,Fuel Type,Rated Efficiency,Annual Run Hours,Generation Capacity,Spec Sheet Link,Model Used,Duplicate Equipment Documents,Latest Facility Filename
0,AD3S008F_3,Success,2025-11-20,Grayson Lumber Corporation,"505 County Road 94, Houston, AL 35572",Houston,AL,NaN,Winston County,NaN,...,14.77,MMBtu/hr,Wood,NaN,8760,NaN,NaN,NaN,1.0,AD3S008F_3
1,AD3S008F_3,Success,2025-11-20,Grayson Lumber Corporation,"505 County Road 94, Houston, AL 35572",Houston,AL,NaN,Winston County,NaN,...,175,MBF,NaN,NaN,8760,NaN,NaN,NaN,1.0,AD3S008F_3
2,AD3S008F_3,Success,2025-11-20,Grayson Lumber Corporation,"505 County Road 94, Houston, AL 35572",Houston,AL,NaN,Winston County,NaN,...,NaN,NaN,NaN,NaN,8760,NaN,NaN,NaN,1.0,AD3S008F_3
3,AD3S008F_3,Success,2025-11-20,Grayson Lumber Corporation,"505 County Road 94, Houston, AL 35572",Houston,AL,NaN,Winston County,NaN,...,NaN,NaN,NaN,NaN,8760,NaN,NaN,NaN,1.0,AD3S008F_3
4,60294_TV_Permit,Success,2025-11-20,Naval Air Station Oceana,"1750 Tomcat Boulevard, Virginia Beach, VA 2346...",Virginia Beach,VA,23460-5120,NaN,928110,...,2.0 MMBtu/hr,NaN,Natural Gas,NaN,NaN,NaN,NaN,NaN,1.0,60294_TV_Permit


In [3]:
raw_df.shape, raw_df.columns.tolist()

((54367, 37),
 ['Filename',
  'Status',
  'Processing Date',
  'Facility Name',
  'Facility Address',
  'Facility City',
  'Facility State Abbreviation',
  'Facility Zip Code',
  'Facility County',
  'NAICS Code',
  'Operating Hours',
  'Industry Description',
  'Permit Number',
  'Issuance Date',
  'Expiration Date',
  'Regulatory Authority',
  'Primary Applicable Regulations (e.g., Title V, PSD, NESHAP Subpart)',
  'Unit ID',
  'Unit Description',
  'Unit Quantity',
  'Unit Make',
  'Unit Model',
  'Year of Manufacture',
  'Unit Type',
  'Pollutants',
  'Emission Limits',
  'Control Device(s)',
  'Capacity Value',
  'Capacity Unit',
  'Fuel Type',
  'Rated Efficiency',
  'Annual Run Hours',
  'Generation Capacity',
  'Spec Sheet Link',
  'Model Used',
  'Duplicate Equipment Documents',
  'Latest Facility Filename'])

## Facilities and states

In [4]:
df = raw_df.copy()

facility_cols = [
    "Facility Name",
    "Facility Address",
    "Facility City",
    "Facility State Abbreviation",
    "Facility Zip Code",
]

def build_facility_key(frame, columns):
    cols = [c for c in columns if c in frame.columns]
    if not cols:
        return pd.Series(["" for _ in range(len(frame))], index=frame.index)

    cleaned = frame[cols].fillna("").astype(str)
    for col in cols:
        cleaned[col] = cleaned[col].str.strip()
    key = cleaned.agg(" | ".join, axis=1)
    return key


def non_empty(series):
    return series.notna() & series.astype(str).str.strip().ne("")


df["facility_key"] = build_facility_key(df, facility_cols)

df["facility_key"] = (
    df["facility_key"]
    .str.replace(r"(\s*\|\s*)+", " | ", regex=True)
    .str.strip(" |")
)

In [5]:
state_col = "Facility State Abbreviation"

valid_facility_mask = df["facility_key"].ne("")
unique_facilities = df.loc[valid_facility_mask, "facility_key"].nunique()

if state_col in df.columns:
    states_with_data = (
        df[state_col]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", np.nan)
        .dropna()
    )
    unique_states = states_with_data.nunique()
else:
    unique_states = 0

unique_facilities, unique_states

(12110, 53)

In [7]:
if state_col in df.columns:
    facilities_by_state = (
        df.loc[valid_facility_mask, ["facility_key", state_col]]
        .dropna(subset=[state_col])
        .assign(**{state_col: lambda x: x[state_col].astype(str).str.strip()})
        .loc[lambda x: x[state_col].ne("")]
        .drop_duplicates()
        .groupby(state_col)["facility_key"]
        .nunique()
        .sort_values(ascending=False)
    )
    facilities_by_state.head(15)
else:
    pd.Series(dtype=int)

In [8]:
naics_col = "NAICS Code"

if naics_col in df.columns:
    naics_series = (
        df[naics_col]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", np.nan)
        .dropna()
    )
    unique_naics = naics_series.nunique()
else:
    unique_naics = 0

unique_naics

985

In [9]:
if naics_col in df.columns:
    naics_counts = naics_series.value_counts().head(15)
    naics_counts
else:
    pd.Series(dtype=int)

In [10]:
coverage_fields = [
    "Facility Name",
    "Facility Address",
    "Facility City",
    "Facility State Abbreviation",
    "Facility Zip Code",
    "Facility County",
    "NAICS Code",
    "Industry Description",
    "Permit Number",
    "Issuance Date",
    "Expiration Date",
    "Regulatory Authority",
    "Primary Applicable Regulations (e.g., Title V, PSD, NESHAP Subpart)",
    "Unit ID",
    "Unit Description",
    "Unit Type",
    "Pollutants",
    "Emission Limits",
    "Control Device(s)",
]

available_coverage_fields = [c for c in coverage_fields if c in df.columns]

coverage = {
    field: non_empty(df[field]).mean() for field in available_coverage_fields
}

pd.Series(coverage).sort_values(ascending=False)

Facility Name                                                          0.999761
Facility State Abbreviation                                            0.982250
Regulatory Authority                                                   0.961760
Unit Description                                                       0.948020
Unit ID                                                                0.943753
Facility City                                                          0.939798
Permit Number                                                          0.906984
Facility Address                                                       0.881656
Industry Description                                                   0.748800
Facility Zip Code                                                      0.742436
Issuance Date                                                          0.713024
Facility County                                                        0.697703
Expiration Date                         

## Field coverage

In [11]:
coverage_by_state_fields = [
    "Facility Address",
    "Facility City",
    "Facility Zip Code",
    "NAICS Code",
    "Industry Description",
    "Permit Number",
]

if state_col in df.columns:
    coverage_by_state_fields = [
        c for c in coverage_by_state_fields if c in df.columns
    ]
    state_df = df[df[state_col].notna()].copy()
    state_df[state_col] = state_df[state_col].astype(str).str.strip()
    state_df = state_df[state_df[state_col] != ""]

    state_coverage = (
        state_df
        .groupby(state_col)[coverage_by_state_fields]
        .apply(lambda g: g.apply(non_empty).mean())
        .sort_values(by=coverage_by_state_fields[0], ascending=False)
    )
    state_coverage.head(15)
else:
    pd.DataFrame()

## Coverage by state

In [13]:
units_per_facility = (
    df.loc[valid_facility_mask]
    .groupby("facility_key")
    .size()
    .rename("unit_rows")
)

units_per_facility.value_counts().head(15)

unit_rows
1     3657
2     1136
4     1132
3     1105
5     1029
6      951
7      765
8      607
9      465
10     394
11     267
12     197
13     145
14     102
15      55
Name: count, dtype: int64

## Units per facility

## Maps

In [16]:
import plotly.express as px

state_counts = (
    df.loc[valid_facility_mask, ["facility_key", state_col]]
    .dropna(subset=[state_col])
    .assign(**{state_col: lambda x: x[state_col].astype(str).str.strip()})
    .loc[lambda x: x[state_col].ne("")]
    .drop_duplicates()
    .groupby(state_col)["facility_key"]
    .nunique()
    .reset_index(name="facility_count")
)

fig = px.choropleth(
    state_counts,
    locations=state_col,
    locationmode="USA-states",
    color="facility_count",
    scope="usa",
    color_continuous_scale="Blues",
    labels={"facility_count": "Facilities"},
    title="Facilities by state",
)
fig.show()

In [ ]:
if naics_col in df.columns:
    naics_coverage_by_state = (
        df[[state_col, naics_col]]
        .copy()
        .assign(
            **{
                state_col: lambda x: x[state_col].astype(str).str.strip(),
                naics_col: lambda x: x[naics_col].astype(str).str.strip(),
            }
        )
        .loc[lambda x: x[state_col].ne("")]
        .groupby(state_col)[naics_col]
        .apply(lambda s: non_empty(s).mean())
        .reset_index(name="naics_coverage")
    )

    fig = px.choropleth(
        naics_coverage_by_state,
        locations=state_col,
        locationmode="USA-states",
        color="naics_coverage",
        scope="usa",
        color_continuous_scale="Viridis",
        labels={"naics_coverage": "NAICS coverage"},
        title="NAICS coverage by state",
        range_color=(0, 1),
    )
    fig.show()
else:
    pd.DataFrame()